# Greyscale ablation — Variant C / Design C1 (thin orchestrator)

**Isolated experiment on branch `greyscale-experiment`.** Adds NEW files only; touches
no completed Phase 1–4 config, module, checkpoint, report, or figure.

## What Design C1 tests
The completed study found that zero-shot cross-modality DR grading **collapses**
(96% grade 0 on OLIVES), and that the confident "severe" tail is an **image-brightness
artifact** — the colour EyePACS grader keys off intensity/colour cues that near-IR OLIVES
renders very differently. Design C1 asks: *if we harmonise the source toward the target
intensity distribution, does the collapse ease?*

**Variant C transform (EyePACS only):** extract the **green** channel (most feature-rich for
retinal lesions/vessels) → **histogram-match** it to the OLIVES near-IR intensity distribution
→ **replicate** into 3 identical channels (so the 3-channel ResNet input still fits).

**Design C1:** transform **EyePACS only**. OLIVES is left **raw** (near-IR) at both train and
eval time. The goal is "make the source look like the target."

**Label preservation is a hard requirement:** the transform modifies ONLY the image tensor;
the DR grade label passes through unchanged. We reuse the existing EyePACS dataset, its
labels, and the existing split (seed 42) — no new label pipeline, no reordering, no drops.

This notebook is a **thin orchestrator**: mount Drive → introspection notes → compute the
OLIVES reference histogram → prove label preservation → **stop**. Training and evaluation
are Prompt 2.

## STEP 1 — how Phase 2 training runs, and where the new transform hooks in

**Training entry point.** `src/training/train.py` (`python -m src.training.train --config
configs/train.yaml [--smoke]`). It merges `configs/model.yaml` + `configs/data.yaml` +
`configs/train.yaml` (`build_merged_config`), applies `loss_weight_overrides`, sets seed 42,
builds loaders via `src/data/dataloaders.py::build_dataloaders`, builds `DRModel.from_config`,
and runs `src/training/trainer.py::Trainer.fit()`. Launched from `notebooks/phase2_training.ipynb`
(`!python -m src.training.train --config configs/train.yaml`).

**Checkpoint naming.** `Trainer._ckpt_path` writes `phase2_{run_name}_{best,last}.pt` into
`cfg.checkpoint.dir`. Phase 2 used `run_name: coral_on` → `phase2_coral_on_best.pt`. Our
`configs/greyscale_train.yaml` uses `run_name: greenhistmatch` and
`checkpoint.dir: .../greyscale_experiment` → `phase2_greenhistmatch_best.pt` (never the
original name). Model selection = validation DR QWK.

**EyePACS dataset + transforms.** `src/data/datasets.py::EyePACSDataset.__getitem__` reads a
stored image tensor, `image = self.transform(image)` if a transform is set, then returns
`{images, dataset:0, dr_labels:int(self._labels[global_idx]), ...}`. **Labels come from the
immutable `self._labels` vector, independent of pixel values** — the transform is the exact,
clean insertion point and cannot affect labels. `build_dataloaders` applies the SAME
`train_tf` to both EyePACS and OLIVES, so EyePACS-only harmonisation is done by wrapping the
EyePACS sub-dataset's `.transform` after the bundle is built.

**Stored value range + normalisation order.** `src/data/preprocess.py::_build_transform`
saves both datasets ImageNet-**normalised**: `(img/255 − mean)/std`. `src/data/transforms.py`
augments by de-normalise → op in [0,1] → clamp → re-normalise. So harmonisation must run on
the **de-normalised [0,1] image before ImageNet normalisation**. `GreenHistMatch` mirrors
this: de-normalise → green + histogram-match + 3ch replicate → re-normalise, composed
**before** the existing augmentation.

**OLIVES loader + range (for the reference histogram).** `configs/data.yaml::paths.olives_file`
→ `preprocessed/olives/olives_fundus.pt`, loaded by `OLIVESDataset` (and the eval loader in
`zero_shot_grading.py`). Images are `[N,3,224,224]` ImageNet-normalised near-IR;
`scripts/compute_olives_histogram.py` de-normalises the green channel to [0,1] and aggregates
the global intensity CDF — the target `GreenHistMatch` matches EyePACS toward.

**Where the new transform hooks in (Prompt 2).** A NEW training wrapper will call the existing
`build_dataloaders(cfg)` and then
`src.preprocessing.harmonise.harmonise_eyepacs_in_bundle(bundle, cfg.reference_hist,
cfg.imagenet_mean, cfg.imagenet_std, cfg.reference_channel)`, which wraps ONLY the EyePACS
sub-datasets (identified by exposing `.labels`), leaving OLIVES raw. `train.py`,
`dataloaders.py`, and `transforms.py` are NOT modified.

In [ ]:
# Setup: mount Drive, restore the repo, cd in. (Mirrors phase2_training.ipynb.)
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/dr-dissertation'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
%cd {REPO_DIR}
# Use the greyscale-experiment branch (NEW files live here).
!git fetch origin && git checkout greyscale-experiment && git pull

import torch
print('CUDA available:', torch.cuda.is_available())

## 1. Compute the OLIVES near-IR reference histogram
One-off, CPU-fine, idempotent. Writes
`greyscale_experiment/olives_reference_hist.npy` (green-channel intensity CDF in [0,1]).

In [ ]:
!python -m scripts.compute_olives_histogram --config configs/greyscale_train.yaml

## 2. Label-preservation verification (MANDATORY — before any training)
Prove the transform is image-only:
1. sample batch **with vs without** harmonise → assert label tensors identical element-wise;
2. per-grade label counts (0–4) over the **full** harmonised train split == original counts;
3. before/after thumbnails (colour vs green-hist-matched).

If (1) or (2) fail, **STOP** — labels are not being preserved.

In [ ]:
# Build the exact Phase-2 loaders (seed 42, same split) via the unmodified pipeline.
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.training.train import build_merged_config
from src.data.dataloaders import build_dataloaders, collate_fn
from src.preprocessing.harmonise import GreenHistMatch, harmonise_eyepacs_in_bundle

cfg = build_merged_config('configs/greyscale_train.yaml')
bundle = build_dataloaders(cfg)

# EyePACS train sub-dataset (ConcatDataset([eyepacs_ds, olives_ds]).datasets[0]).
ep_ds = bundle['train_loader'].dataset.datasets[0]
train_idx = bundle['splits']['eyepacs']['train']

# ORIGINAL per-grade counts — ground truth from the immutable label vector.
orig_labels = ep_ds.labels[torch.as_tensor(list(train_idx))].long()
orig_counts = torch.bincount(orig_labels, minlength=5)
print('EyePACS train split size:', len(ep_ds))
print('original per-grade counts (0-4):', orig_counts.tolist())

In [ ]:
# Capture a few RAW normalised tensors (transform off) for clean thumbnails + a
# confound-free pixel-change check, then restore the original transform.
sample_ids = list(range(6))
orig_tf = ep_ds.transform
ep_ds.transform = None
raw_norm = [ep_ds[i]['images'].clone() for i in sample_ids]
raw_labels = [ep_ds[i]['dr_labels'] for i in sample_ids]
ep_ds.transform = orig_tf

ghm = GreenHistMatch(
    cfg.reference_hist, cfg.imagenet_mean, cfg.imagenet_std, int(cfg.reference_channel)
)
harm_norm = [ghm(t) for t in raw_norm]

# Harmonisation must change pixels (image side) but not the label passed alongside.
pixel_delta = float(torch.stack([(h - r).abs().mean() for h, r in zip(harm_norm, raw_norm)]).mean())
print('mean abs pixel change (harmonised vs colour):', pixel_delta)
assert pixel_delta > 0.0, 'transform did not change pixels — unexpected'

In [ ]:
# (1) Sample batch WITH vs WITHOUT harmonise -> labels identical element-wise.
sample_n = 128
labels_before = torch.tensor([ep_ds[i]['dr_labels'] for i in range(sample_n)])

# Turn on Design C1 harmonisation for EyePACS ONLY (OLIVES untouched).
n_wrapped = harmonise_eyepacs_in_bundle(
    bundle, cfg.reference_hist, cfg.imagenet_mean, cfg.imagenet_std, int(cfg.reference_channel)
)
print('EyePACS datasets harmonised (train/val/test):', n_wrapped)

labels_after = torch.tensor([ep_ds[i]['dr_labels'] for i in range(sample_n)])
assert torch.equal(labels_before, labels_after), 'LABELS CHANGED — STOP (labels not preserved)'
print('sample-batch labels identical with vs without harmonise:', True)

In [ ]:
# (2) Full harmonised train split -> per-grade counts must equal the original.
# ep_ds is now harmonised; iterate the FULL split through the real pipeline and
# collect only labels (images are transformed and discarded).
ep_loader = DataLoader(
    ep_ds, batch_size=256, shuffle=False, collate_fn=collate_fn, num_workers=2
)
harm_counts = torch.zeros(5, dtype=torch.long)
for b in tqdm(ep_loader, desc='harmonised EyePACS train'):
    harm_counts += torch.bincount(b['dr_labels'], minlength=5)

print('original    per-grade counts (0-4):', orig_counts.tolist())
print('harmonised  per-grade counts (0-4):', harm_counts.tolist())
assert torch.equal(harm_counts, orig_counts), (
    'PER-GRADE COUNTS DIFFER — STOP (labels not preserved over the full split)'
)
print('\nLABEL PRESERVATION VERIFIED: image-only transform, labels + split intact.')

In [ ]:
# (3) Before/after thumbnails: colour EyePACS vs green-hist-matched (eyeball check).
import matplotlib.pyplot as plt

mean = torch.tensor(list(cfg.imagenet_mean)).view(3, 1, 1)
std = torch.tensor(list(cfg.imagenet_std)).view(3, 1, 1)

def denorm(t):
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

n = len(sample_ids)
fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5))
for j, (r, h, lab) in enumerate(zip(raw_norm, harm_norm, raw_labels)):
    axes[0, j].imshow(denorm(r)); axes[0, j].axis('off')
    axes[0, j].set_title(f'colour (grade {lab})', fontsize=9)
    axes[1, j].imshow(denorm(h)); axes[1, j].axis('off')
    axes[1, j].set_title('green + hist-match', fontsize=9)
fig.suptitle('EyePACS source: colour (top) vs Design C1 harmonised (bottom)', y=1.02)
plt.tight_layout(); plt.show()

## Stop here — training & evaluation are Prompt 2

If both assertions passed and the thumbnails look like plausible retinal images (vessels/optic
disc preserved, intensity remapped toward near-IR), the harmonisation is verified as
image-only and label-preserving.

**Prompt 2 will:**
- train with `python -m <new wrapper> --config configs/greyscale_train.yaml` (EyePACS-only
  harmonisation via `harmonise_eyepacs_in_bundle`; checkpoint → `phase2_greenhistmatch_best.pt`);
- evaluate with the unmodified
  `python -m src.analysis.zero_shot_grading --config configs/greyscale_eval.yaml`
  (OLIVES raw; outputs under `greyscale_experiment/`).

Nothing above overwrites `phase2_coral_on_best.pt` or any existing report/figure.